In [1]:

import numpy as np
import random

# Environment parameters
GRID_SIZE = 50
TRASH = 1
EMPTY = 0
WALL = -1

# Actions
ACTIONS = {
    0: "MOVE_N",
    1: "MOVE_S",
    2: "MOVE_E",
    3: "MOVE_W",
    4: "MOVE_RANDOM",
    5: "STAY",
    6: "COLLECT_TRASH"
}

NUM_ACTIONS = len(ACTIONS)

# Rewards
REWARD_COLLECT_SUCCESS = 10
REWARD_COLLECT_FAIL = -1
REWARD_WALL_COLLISION = -5

# Q-Learning parameters
ALPHA = 0.1  # Learning rate
GAMMA = 0.9  # Discount factor
EPSILON_START = 1.0  # Initial epsilon for epsilon-greedy
EPSILON_END = 0.01   # Final epsilon
EPSILON_DECAY = 0.995 # Epsilon decay rate

NUM_EPISODES = 500
MAX_STEPS_PER_EPISODE = 1000

class MarvinEnvironment:
    def __init__(self, grid_size=GRID_SIZE):
        self.grid_size = grid_size
        self.grid = self._generate_random_grid()
        self.marvin_pos = (0, 0) # Marvin always starts at (0,0)
        self.current_episode_score = 0

    def _generate_random_grid(self):
        grid = np.zeros((self.grid_size, self.grid_size), dtype=int)
        # Place some trash and walls randomly
        for _ in range(self.grid_size * self.grid_size // 10): # 10% trash
            r, c = random.randint(0, self.grid_size - 1), random.randint(0, self.grid_size - 1)
            if (r, c) != (0, 0): # Don't place trash at start
                grid[r, c] = TRASH
        for _ in range(self.grid_size * self.grid_size // 20): # 5% walls
            r, c = random.randint(0, self.grid_size - 1), random.randint(0, self.grid_size - 1)
            if (r, c) != (0, 0) and grid[r, c] != TRASH: # Don't place wall at start or on trash
                grid[r, c] = WALL
        return grid

    def get_state(self):
        r, c = self.marvin_pos
        current_cell_content = self.grid[r, c]

        # Get adjacent cell contents (N, S, E, W)
        north = self.grid[r-1, c] if r > 0 else WALL
        south = self.grid[r+1, c] if r < self.grid_size - 1 else WALL
        east = self.grid[r, c+1] if c < self.grid_size - 1 else WALL
        west = self.grid[r, c-1] if c > 0 else WALL

        # State representation: (current_cell, north, south, east, west)
        # We need to map these values to a discrete state space for the Q-table
        # Since content can be -1, 0, 1, we can use a tuple as state key
        return (current_cell_content, north, south, east, west)

    def reset(self):
        self.marvin_pos = (0, 0)
        self.grid = self._generate_random_grid() # New environment for each episode
        self.current_episode_score = 0
        return self.get_state()

    def step(self, action):
        r, c = self.marvin_pos
        new_r, new_c = r, c
        reward = 0
        done = False

        if action == ACTIONS[0]: # MOVE_N
            new_r -= 1
        elif action == ACTIONS[1]: # MOVE_S
            new_r += 1
        elif action == ACTIONS[2]: # MOVE_E
            new_c += 1
        elif action == ACTIONS[3]: # MOVE_W
            new_c -= 1
        elif action == ACTIONS[4]: # MOVE_RANDOM
            move_options = [(r-1, c), (r+1, c), (r, c+1), (r, c-1)]
            # Filter out-of-bounds moves for random action
            valid_move_options = []
            for mr, mc in move_options:
                if 0 <= mr < self.grid_size and 0 <= mc < self.grid_size:
                    valid_move_options.append((mr, mc))
            if valid_move_options: # Only choose if there are valid options
                new_r, new_c = random.choice(valid_move_options)
            else:
                # If no valid random moves, stay put and potentially incur wall penalty if current is wall
                pass
        elif action == ACTIONS[5]: # STAY
            pass # Position remains the same
        elif action == ACTIONS[6]: # COLLECT_TRASH
            if self.grid[r, c] == TRASH:
                reward = REWARD_COLLECT_SUCCESS
                self.grid[r, c] = EMPTY # Trash collected
            else:
                reward = REWARD_COLLECT_FAIL
            self.current_episode_score += reward
            return self.get_state(), reward, done # No movement, so state is current

        # Handle movement actions
        if not (0 <= new_r < self.grid_size and 0 <= new_c < self.grid_size) or \
           self.grid[new_r, new_c] == WALL:
            reward = REWARD_WALL_COLLISION
            # Marvin stays in current position if collision
        else:
            self.marvin_pos = (new_r, new_c)

        self.current_episode_score += reward
        return self.get_state(), reward, done

class QLearningAgent:
    def __init__(self, num_actions=NUM_ACTIONS, alpha=ALPHA, gamma=GAMMA, 
                 epsilon_start=EPSILON_START, epsilon_end=EPSILON_END, 
                 epsilon_decay=EPSILON_DECAY):
        self.q_table = {}
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

    def get_q_value(self, state, action_idx):
        return self.q_table.get(state, np.zeros(self.num_actions))[action_idx]

    def choose_action(self, state):
        # Ensure the state exists in q_table before choosing action
        if state not in self.q_table:
            self.q_table[state] = np.zeros(self.num_actions)

        if random.uniform(0, 1) < self.epsilon:
            return random.choice(list(ACTIONS.values())) # Explore
        else:
            # Exploit: choose action with max Q-value
            q_values = self.q_table[state]
            
            # If all Q-values are the same (e.g., all zeros for a new state),
            # np.where will return all indices. random.choice will then pick one.
            max_q = np.max(q_values)
            best_actions_indices = np.where(q_values == max_q)[0].tolist() # Convert to list
            
            # This check is a safeguard, though with proper initialization it should not be needed.
            if not best_actions_indices: # Check if the list is empty
                return random.choice(list(ACTIONS.values())) # Fallback to random if no best action found

            chosen_action_idx = random.choice(best_actions_indices)
            return ACTIONS[chosen_action_idx]

    def learn(self, state, action, reward, next_state):
        action_idx = list(ACTIONS.values()).index(action)
        
        if state not in self.q_table:
            self.q_table[state] = np.zeros(self.num_actions)
        if next_state not in self.q_table:
            self.q_table[next_state] = np.zeros(self.num_actions)

        old_value = self.q_table[state][action_idx]
        next_max = np.max(self.q_table[next_state])

        new_value = old_value + self.alpha * (reward + self.gamma * next_max - old_value)
        self.q_table[state][action_idx] = new_value

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)


def train_marvin():
    env = MarvinEnvironment()
    agent = QLearningAgent()
    
    episode_scores = []

    for episode in range(NUM_EPISODES):
        state = env.reset()
        done = False
        
        for step in range(MAX_STEPS_PER_EPISODE):
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.learn(state, action, reward, next_state)
            state = next_state
            
            # If the environment has a concept of 'done' (e.g., all trash collected)
            # if done:
            #     break
        
        episode_scores.append(env.current_episode_score)
        agent.decay_epsilon()
        
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode + 1}/{NUM_EPISODES}, Score: {env.current_episode_score}, Epsilon: {agent.epsilon:.4f}")
            
    print("\nTraining finished.")
    return episode_scores, agent.q_table, env

if __name__ == "__main__":
    scores, q_table, final_env = train_marvin()
    
    # Save scores for plotting
    np.save("episode_scores.npy", np.array(scores))
    
    print("\nFirst 10 episode scores:", scores[:10])
    print("Last 10 episode scores:", scores[-10:])
    
    # Example of optimal policy for a few states (for report)
    print("\nOptimal policy for some random states:")
    # Select some states from the q_table that were actually visited
    visited_states = list(q_table.keys())
    if len(visited_states) > 5:
        sample_states = random.sample(visited_states, 5)
    else:
        sample_states = visited_states

    for state in sample_states:
        q_values = q_table[state]
        optimal_action_idx = np.argmax(q_values)
        optimal_action = ACTIONS[optimal_action_idx]
        print(f"State {state}: Optimal Action -> {optimal_action}")

    # You can also visualize the Q-table or policy for a small part of the grid
    # For a 50x50 grid, visualizing the full policy is complex.
    # Instead, we can show the policy for a small, fixed environment.
    # This part would be better for the report, possibly by creating a small test environment.

    # To demonstrate policy on a small grid for the report:
    # Create a small, fixed grid for visualization purposes
    small_grid_env = MarvinEnvironment(grid_size=5) # Example 5x5 grid
    small_grid_env.grid = np.array([
        [0, 0, TRASH, 0, 0],
        [0, WALL, 0, WALL, 0],
        [0, 0, 0, TRASH, 0],
        [0, WALL, 0, WALL, 0],
        [TRASH, 0, 0, 0, 0]
    ])
    
    print("\nOptimal policy for a small 5x5 test environment:")
    for r in range(small_grid_env.grid_size):
        for c in range(small_grid_env.grid_size):
            small_grid_env.marvin_pos = (r, c)
            state = small_grid_env.get_state()
            if state in q_table:
                q_values = q_table[state]
                optimal_action_idx = np.argmax(q_values)
                optimal_action = ACTIONS[optimal_action_idx]
                print(f"Pos ({r},{c}), State {state}: Optimal Action -> {optimal_action}")
            else:
                print(f"Pos ({r},{c}), State {state}: Not visited during training or no optimal action found.")

Episode 50/500, Score: -85, Epsilon: 0.7783
Episode 100/500, Score: 177, Epsilon: 0.6058
Episode 150/500, Score: 145, Epsilon: 0.4715
Episode 200/500, Score: 191, Epsilon: 0.3670
Episode 250/500, Score: 220, Epsilon: 0.2856
Episode 300/500, Score: 154, Epsilon: 0.2223
Episode 350/500, Score: 396, Epsilon: 0.1730
Episode 400/500, Score: 328, Epsilon: 0.1347
Episode 450/500, Score: 401, Epsilon: 0.1048
Episode 500/500, Score: 323, Epsilon: 0.0816

Training finished.

First 10 episode scores: [-342, -251, -385, -165, -170, -304, -275, -224, -405, -142]
Last 10 episode scores: [390, 362, 202, 409, 517, 466, 302, 243, 307, 323]

Optimal policy for some random states:
State (0, -1, 1, 0, -1): Optimal Action -> MOVE_S
State (0, 1, 0, -1, -1): Optimal Action -> MOVE_N
State (0, 0, 0, -1, 1): Optimal Action -> MOVE_W
State (0, 0, 0, 1, 0): Optimal Action -> MOVE_E
State (1, 1, -1, -1, 0): Optimal Action -> MOVE_RANDOM

Optimal policy for a small 5x5 test environment:
Pos (0,0), State (0, -1, 0,